In [57]:
import pandas as pd

In [58]:
stk_data=pd.read_csv("Tatacoffee13_21.csv")

In [59]:
stk_data=stk_data[(stk_data["Date"]>="2020-07-01")&(stk_data["Date"]<="2021-12-28")]

In [60]:
stk_data

,Date,Open,High,Low,Close
1851,2020-07-01,81.50,82.70,81.05,81.95
1852,2020-07-02,82.25,82.70,81.55,81.90
1853,2020-07-03,82.05,85.45,82.05,84.35
1854,2020-07-06,85.80,87.85,85.05,86.60
1855,2020-07-07,86.70,87.00,85.05,85.65
...,...,...,...,...,...
2220,2021-12-22,202.90,207.80,201.35,205.00
2221,2021-12-23,206.00,206.85,202.05,202.95
2222,2021-12-24,203.90,203.90,199.35,201.00
2223,2021-12-27,200.00,222.00,196.00,218.35


In [61]:
# preprocessing
from sklearn.preprocessing import MinMaxScaler
Ms=MinMaxScaler()
data1=Ms.fit_transform(stk_data[["Open","High","Low","Close"]])
print("Len:",data1.shape)

Len: (374, 4)


In [62]:
data1=pd.DataFrame(data1, columns=['Open','High','Low','Close'])

In [63]:
data1

,Open,High,Low,Close
0,0.015432,0.015482,0.021817,0.025032
1,0.020062,0.015482,0.025073,0.024715
2,0.018827,0.031250,0.028330,0.040241
3,0.041975,0.045011,0.047867,0.054499
4,0.047531,0.040138,0.047867,0.048479
...,...,...,...,...
369,0.764815,0.732798,0.805275,0.804816
370,0.783951,0.727351,0.809834,0.791825
371,0.770988,0.710436,0.792250,0.779468
372,0.746914,0.814220,0.770433,0.889417


In [64]:
# Training and Test data split
training_size=round(len(data1)*0.80)
print(training_size)
X_train=data1[:training_size]
X_test=data1[training_size:]
print("X_train length:",X_train.shape)
print("X_test length:",X_test.shape)
Y_train=data1[:training_size]
Y_test=data1[training_size:]
print("Y_train length:", Y_train.shape)
print("Y_test length:",Y_test.shape)


299
X_train length: (299, 4)
X_test length: (75, 4)
Y_train length: (299, 4)
Y_test length: (75, 4)


In [65]:
import warnings
warnings.filterwarnings("ignore")

In [66]:
# Creating a dictionary 
performance={"Model":[],"RMSE":[],"MaPe":[],"Lag":[],"Test":[]}

In [67]:
# Defining the function cominbation with two parameters, dataset(values) and list(columnnames:keys) to be passed while calling the function
def cominbation(dataset,listt):
    # print the listt(i.e, key values or column names)
    print(listt)
    # Creating the table with passed keys and values i.e data and column names
    datasetTwo=dataset[listt]
    print(datasetTwo)
    # Here we are determining the number of samples we are going to take for test and based on that training samples are calculated
    test_obs=28
    # Subtracting test samples from overall data to determine the training samples
    train=datasetTwo[:-test_obs]
    # Test sample determined is taken for testing(here its 28)
    test=datasetTwo[-test_obs:]
    print("Train length:",train.shape)
    print("Test length:", test.shape)
    # import VAR module from statsmodels
    from statsmodels.tsa.api import VAR
    # here we have the lag orders from 1 to 10 to find aic and bic for each lag-order
    for i in [1,2,3,4,5,6,7,8,9,10]:
    # passing the training data to the model
        model=VAR(train)
        results=model.fit(i)
        print("Order=", i)
        print("AIC:",results.aic)
        print("BIC:",results.bic)
    # select_order is used to test the model for each lag and keeping max lag as 12
    x=model.select_order(maxlags=12)
    # here we want to select the lag which has lowest aic value
    order=x.selected_orders["aic"]
     # Fitting the model with the best order and storing the results of them in result
    result = model.fit(order)
    result.summary()
    # Here we taking the lagged_values from training data based on the selected order above(if the order is selected as 1 then only the last row of the 
    # training data will be taken to lagged_values)
    lagged_Values=train.values[-order:]
    print(lagged_Values)
    print("order =", order)
    print("train values shape =",train.values.shape)
    print("train shape =", train.shape)
    print("lagged_values shape =", lagged_Values.shape)
    # Here we got shape mismatch issue as the forecast expects 2 d array and the lagged values are 1 d array as we have 4 variables we  have re-shaped
    # this way so the forecast method doesn't throw a value error
    lagged_Values = lagged_Values.reshape(order, train.shape[1])
    print("order =", order)
    print("result.k_ar =", result.k_ar)
    print("train shape =", train.shape)
    print("lagged_values shape =", lagged_Values.shape)
    # for the taken lagged_values(training input) we are predicting the next 28 values in the time series
    pred=result.forecast(y=lagged_Values, steps=28)
    # Creating the table for the predicted series 
    preds=pd.DataFrame(pred,columns=listt)
    print(preds)
    # saving the result to csv file
    preds.to_csv("varforecasted_{}.csv".format(test_obs))
    # evaluating the model using mean_square_error
    from sklearn.metrics import mean_squared_error
    # passing test data and training data to determine rmse
    rmse= round(mean_squared_error(test,pred,squared=False),4)
    # mean absolute percentage error to check the accuracy of the model
    from sklearn.metrics import mean_absolute_percentage_error
    # passing the test data values and predicted values to determine the prediction accuracy
    mape=mean_absolute_percentage_error(test,pred)
    # Now appending the list with the values obtained one by one, i.e model(variables used to train the model)
    # RMSE - root mean square for each combination we give
    # Mape- for each combination we give
    # Lag- which lag is selected for each combination
    # Test- how many test observation are used - here it is 28
    performance["Model"].append(listt)
    performance["RMSE"].append(rmse)
    performance["MaPe"].append(mape)
    performance["Lag"].append(order)
    performance["Test"].append(test_obs)
    perf=pd.DataFrame(performance)
    return perf,result,pred

In [68]:
# TO get all the combinations in one function the below code can be used
listt = [
    ['Close', 'High'],
    ['Close', 'High', 'Open'],
    ['Close', 'High', 'Open', 'Low']
]

for combination in listt:
    perf, result, pred = cominbation(data1, combination)

['Close', 'High']
        Close      High
0    0.025032  0.015482
1    0.024715  0.015482
2    0.040241  0.031250
3    0.054499  0.045011
4    0.048479  0.040138
..        ...       ...
369  0.804816  0.732798
370  0.791825  0.727351
371  0.779468  0.710436
372  0.889417  0.814220
373  0.851394  0.805333

[374 rows x 2 columns]
Train length: (346, 2)
Test length: (28, 2)
Order= 1
AIC: -16.105309866465785
BIC: -16.038465615734808
Order= 2
AIC: -16.21327111752211
BIC: -16.101624557714743
Order= 3
AIC: -16.197908668310685
BIC: -16.041266609242687
Order= 4
AIC: -16.201617469235938
BIC: -15.99978532518001
Order= 5
AIC: -16.218974759316648
BIC: -15.971756534975777
Order= 6
AIC: -16.207183347961475
BIC: -15.914381624261871
Order= 7
AIC: -16.2021549151692
BIC: -15.863570834870046
Order= 8
AIC: -16.187579554534366
BIC: -15.803012807651461
Order= 9
AIC: -16.172416633412645
BIC: -15.74166544245303
Order= 10
AIC: -16.153474205957682
BIC: -15.676335310962282
[[0.89797212 0.82712156]
 [0.90430925 0.

In [69]:
perf

,Model,RMSE,MaPe,Lag,Test
0,"[Close, High]",0.1656,0.200617,5,28
1,"[Close, High, Open]",0.1612,0.193862,1,28
2,"[Close, High, Open, Low]",0.1587,0.188550,1,28
